# Notebook 01 — Setup & Loughran-McDonald Word List
**Project:** RBI Communication, Financial Stability and Macroeconomic Transmission: Evidence from Sentiment Analysis of MPC Minutes (2010–2025)

This notebook sets up the environment, loads the Loughran-McDonald Finance Dictionary, and prepares word lists for sentiment analysis.

In [1]:
# Install required packages (run once)
import subprocess
packages = [
    'PyPDF2', 'requests', 'pandas', 'numpy', 'openpyxl',
    'pysentiment2', 'yfinance', 'statsmodels', 'matplotlib',
    'seaborn', 'scipy', 'scikit-learn'
]
for pkg in packages:
    subprocess.run(['pip', 'install', pkg, '-q'], capture_output=True)
print("All packages installed successfully.")

All packages installed successfully.


In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── Project paths (always relative — works on any machine) ──────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_RAW     = os.path.join(PROJECT_ROOT, 'data', 'raw')
DATA_PROC    = os.path.join(PROJECT_ROOT, 'data', 'processed')
OUTPUT_FIG   = os.path.join(PROJECT_ROOT, 'output', 'figures')
OUTPUT_TAB   = os.path.join(PROJECT_ROOT, 'output', 'tables')
OUTPUT_RES   = os.path.join(PROJECT_ROOT, 'output', 'results')

for path in [DATA_RAW, DATA_PROC, OUTPUT_FIG, OUTPUT_TAB, OUTPUT_RES]:
    os.makedirs(path, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("All directories ready.")

Project root: C:\Users\rohit\Desktop\Rohit\RBI_Sentiment_Project
All directories ready.


## Load Loughran-McDonald Finance Dictionary
The LM dictionary is the gold standard for financial text sentiment analysis (Loughran & McDonald, 2011, Journal of Finance). It contains finance-specific positive and negative words — far more accurate than generic dictionaries like VADER for central bank documents.

In [3]:
import pysentiment2 as ps

# Load LM dictionary via pysentiment2
lm_model = ps.LM()

# Extract word sets (stored as stemmed tokens internally)
lm_positive = set(lm_model._posset)
lm_negative = set(lm_model._negset)

print(f"LM Positive words : {len(lm_positive):,}")
print(f"LM Negative words : {len(lm_negative):,}")
print()
print("Sample POSITIVE words:", sorted(list(lm_positive))[:15])
print("Sample NEGATIVE words:", sorted(list(lm_negative))[:15])

LM Positive words : 140
LM Negative words : 893

Sample POSITIVE words: ['abund', 'acclaim', 'accomplish', 'achiev', 'adequ', 'advanc', 'advantag', 'allianc', 'assur', 'attain', 'attract', 'beauti', 'benefici', 'benefit', 'better']
Sample NEGATIVE words: ['abandon', 'abdic', 'aberr', 'abet', 'abnorm', 'abolish', 'abrog', 'abrupt', 'abruptli', 'absenc', 'absente', 'abus', 'accid', 'accident', 'accus']


In [4]:
# ── Save LM word lists for use in all other notebooks ─────────────────────
max_len = max(len(lm_positive), len(lm_negative))
pos_sorted = sorted(list(lm_positive))
neg_sorted = sorted(list(lm_negative))

lm_df = pd.DataFrame({
    'Positive_Words': pos_sorted + [''] * (max_len - len(pos_sorted)),
    'Negative_Words': neg_sorted + [''] * (max_len - len(neg_sorted))
})

lm_df.to_csv(os.path.join(DATA_RAW, 'LM_MasterDictionary.csv'), index=False)
print(f"LM dictionary saved: {len(pos_sorted)} positive, {len(neg_sorted)} negative words")
print(f"Saved to: {os.path.join(DATA_RAW, 'LM_MasterDictionary.csv')}")

LM dictionary saved: 140 positive, 893 negative words
Saved to: C:\Users\rohit\Desktop\Rohit\RBI_Sentiment_Project\data\raw\LM_MasterDictionary.csv


## Compare LM vs Custom Dictionary
Your original dictionary had only 295 words each. See the improvement with LM.

In [5]:
# Load original custom dictionary for comparison
custom_path = os.path.join(DATA_RAW, 'positive and negative data.xlsx')
if os.path.exists(custom_path):
    custom_df = pd.read_excel(custom_path)
    custom_pos = set(custom_df['Positive words'].dropna().str.lower().str.strip())
    custom_neg = set(custom_df['Negative words'].dropna().str.lower().str.strip())
    
    print("=" * 50)
    print("DICTIONARY COMPARISON")
    print("=" * 50)
    print(f"{'Source':<20} {'Positive':>10} {'Negative':>10}")
    print("-" * 42)
    print(f"{'Custom (original)':<20} {len(custom_pos):>10,} {len(custom_neg):>10,}")
    print(f"{'LM Finance Dict':<20} {len(lm_positive):>10,} {len(lm_negative):>10,}")
    improvement = len(lm_negative) / len(custom_neg)
    print(f"\nLM has {improvement:.1f}x more negative words — significantly better for finance text.")
else:
    print("Custom dictionary not found — using LM only.")

DICTIONARY COMPARISON
Source                 Positive   Negative
------------------------------------------
Custom (original)            96        295
LM Finance Dict             140        893

LM has 3.0x more negative words — significantly better for finance text.


In [6]:
# ── Quick validation: test on sample RBI sentence ────────────────────────
def lm_sentiment(text):
    tokens = lm_model.tokenize(text)
    score  = lm_model.get_score(tokens)
    return score

test_sentences = [
    "The monetary policy remains accommodative to support growth and recovery.",
    "The banking sector faces elevated stress and adverse credit risks.",
    "Inflation declined and financial conditions improved significantly.",
    "Systemic risks remain elevated amid global uncertainty and volatility."
]

print("LM SENTIMENT VALIDATION")
print("=" * 70)
for s in test_sentences:
    sc = lm_sentiment(s)
    print(f"Text    : {s[:60]}...")
    print(f"Result  : Pos={sc['Positive']}  Neg={sc['Negative']}  Polarity={sc['Polarity']:.3f}")
    print()

LM SENTIMENT VALIDATION
Text    : The monetary policy remains accommodative to support growth ...
Result  : Pos=0  Neg=0  Polarity=0.000

Text    : The banking sector faces elevated stress and adverse credit ...
Result  : Pos=0  Neg=2  Polarity=-1.000

Text    : Inflation declined and financial conditions improved signifi...
Result  : Pos=1  Neg=1  Polarity=0.000

Text    : Systemic risks remain elevated amid global uncertainty and v...
Result  : Pos=0  Neg=1  Polarity=-1.000



In [7]:
print("=" * 50)
print("NOTEBOOK 01 COMPLETE")
print("=" * 50)
print("LM Finance Dictionary loaded and saved.")
print("Ready to proceed to Notebook 02 — Sentiment Extraction.")

NOTEBOOK 01 COMPLETE
LM Finance Dictionary loaded and saved.
Ready to proceed to Notebook 02 — Sentiment Extraction.
